In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import os


def load_results(test_function, strategy, kernel):
    save_dir = f"results/{test_function}/{strategy}+{kernel}"
    ys = []
    ts = []
    for filename in sorted(os.listdir(save_dir)):
        results = np.load(f"{save_dir}/{filename}")
        ys.append(results["ymin"])
        ts.append(results["acquisition_time"])
    return np.array(ys), np.array(ts)


for test_function in ["GoldsteinPrice", "LunarLander", "Push", "Rover"]:
    plt.figure(figsize=(12, 6))
    plt.subplot(1, 2, 1)
    for i, (strategy, color, kernel, linestyle) in enumerate(
        [
            ("LHS", "#1f77b4", "Euclidean+Matern", "solid"),
            ("BFGS", "#ff7f0e", "Euclidean+Matern", "solid"),
            ("Voronoi", "#2ca02c", "Euclidean+Matern", "solid"),
        ]
    ):
        y, t = load_results(test_function, strategy, kernel)
        if y.size == 0:
            continue
        runs, max_acquisitions = y.shape
        x = np.arange(max_acquisitions)

        # ci on median
        qu = np.clip(0.5 + 1.96 * np.sqrt(0.25 / runs), 0, 1)
        ql = np.clip(0.5 - 1.96 * np.sqrt(0.25 / runs), 0, 1)
        ul = np.quantile(y, qu, axis=0)
        ll = np.quantile(y, ql, axis=0)

        plt.plot(
            x,
            np.median(y, axis=0),
            label=f"{strategy} + {kernel} ({runs} runs)",
            color=color,
            linestyle=linestyle,
        )
        plt.fill_between(x, ll, ul, alpha=0.2, color=color)
    plt.title(f"{test_function.capitalize()}")
    plt.xlabel("Number of Acquisitions")
    plt.ylabel("Minimum Function Value")
    plt.legend()
    plt.grid()

    plt.subplot(1, 2, 2)
    strategies = ["LHS", "Voronoi", "BFGS"]
    for i, strategy in enumerate(strategies):
        y, t = load_results(test_function, strategy, kernel)
        if y.size == 0:
            continue
        plt.boxplot(t[:], positions=[i], patch_artist=True)
    plt.xticks(range(len(strategies)), strategies)
    plt.ylabel("Time (seconds)")
    plt.ylim(0, None)
    plt.grid(True)
    plt.show()